In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:

        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/__script__.py
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/platformdirs-4.9.6-py3-none-any.whl
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/shellingham-1.5.4-py2.py3-none-any.whl
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/typing_extensions-4.15.0-py3-none-any.whl
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/httpx-0.28.1-py3-none-any.whl
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/packaging-26.1-py3-none-any.whl
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/h11-0.16.0-py3-none-any.whl
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/certifi-2026.4.22-py3-none-any.whl
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/httpcore-1.0.9-py3-none-any.whl
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/huggingface_hub-1.11.0-py3-none-any.whl
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/click-8.3.3-py3-none-any.whl
/kaggle/input/pm-116354817-at-04-23-2026-12-29-47/fsspec-2026.3.0-py3-none-any.whl
/kaggle/in

In [2]:
from typing import cast

import numpy as np
import torch
import torch.nn as nn
from miditok import REMI, TokSequence, TokenizerConfig
from torch.utils.data import DataLoader, Dataset
import symusic

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_PATH = "/kaggle/input/datasets/anhkhoal0506/tokenised-jazz-music/dataset.npy"   # update to your dataset name
SEED_PATH    = "/kaggle/input/datasets/anhkhoal0506/tokenised-jazz-music/A2_seed.mid"
OUT_PATH     = "/kaggle/working/generated.mid"
MODEL_PATH   = "/kaggle/working/marjazz.pth"

# ── Hyperparameters ───────────────────────────────────────────────────────────
EMBED_DIM   = 128
HIDDEN_SIZE = 256
NUM_LAYERS  = 2
MLP_DIM     = 256
DROPOUT     = 0.3
BATCH_SIZE  = 32
MAX_SEQ_LEN = 1024
EPOCHS      = 400
LR          = 1e-3
LR_FACTOR   = 0.5
LR_PATIENCE = 2
ES_PATIENCE = 5
MAX_GEN_LEN = 500

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = REMI(TokenizerConfig(
    pitch_range=(40, 90),
    special_tokens=["PAD", "BOS", "EOS", "MASK"],
    tempo_range=(80, 160),
    use_pitchdrum_tokens=False,
))
vocab_size = len(tokenizer)
print(f"Vocab size: {vocab_size}")

# ── Dataset ───────────────────────────────────────────────────────────────────
class PreTokenizedDataset(Dataset):
    def __init__(self, path: str):
        self.data = torch.from_numpy(np.load(path, allow_pickle=True).astype(np.int64))

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {"input_ids": self.data[idx]}


def load_pretokenized(path: str = DATASET_PATH, batch_size: int = BATCH_SIZE) -> DataLoader:
    dataset = PreTokenizedDataset(path)
    print(f"Dataset loaded: {len(dataset)} chunks of {MAX_SEQ_LEN} tokens")
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

# ── Model ─────────────────────────────────────────────────────────────────────
class MarJazz(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = EMBED_DIM,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        mlp_dim: int = MLP_DIM,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm1 = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.drop1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(hidden_size)

        self.lstm2 = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.drop2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(hidden_size)

        self.mlp   = nn.Linear(hidden_size, mlp_dim)
        self.act   = nn.GELU()
        self.drop3 = nn.Dropout(dropout)
        self.norm3 = nn.LayerNorm(mlp_dim)

        self.fc = nn.Linear(mlp_dim, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embedding(x)
        x, _ = self.lstm1(x)
        x = self.norm1(self.drop1(x))
        x, _ = self.lstm2(x)
        x = self.norm2(self.drop2(x))
        x = self.act(self.mlp(x))
        x = self.norm3(self.drop3(x))
        return self.fc(x)

# ── Train ─────────────────────────────────────────────────────────────────────
def train(
    model: nn.Module,
    dataloader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = LR,
    lr_factor: float = LR_FACTOR,
    lr_patience: int = LR_PATIENCE,
    es_patience: int = ES_PATIENCE,
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, factor=lr_factor, patience=lr_patience
    )
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    best_loss, no_improve = float("inf"), 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            inputs, targets = input_ids[:, :-1], input_ids[:, 1:]
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(dataloader)
        scheduler.step(avg_loss)
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch+1}/{epochs}  loss={avg_loss:.4f}  lr={current_lr:.2e}")
        if avg_loss < best_loss:
            best_loss, no_improve = avg_loss, 0
        else:
            no_improve += 1
            if no_improve >= es_patience:
                print(f"Early stopping at epoch {epoch+1} (no improvement for {es_patience} epochs)")
                break

# ── Save / Load model ─────────────────────────────────────────────────────────
def save_model(model: nn.Module, path: str = MODEL_PATH):
    torch.save({
        "model_state": model.state_dict(),
        "hyperparams": {
            "vocab_size":   vocab_size,
            "embed_dim":    EMBED_DIM,
            "hidden_size":  HIDDEN_SIZE,
            "num_layers":   NUM_LAYERS,
            "mlp_dim":      MLP_DIM,
            "dropout":      DROPOUT,
        },
    }, path)
    print(f"Model saved → {path}")


def load_model(path: str = MODEL_PATH) -> nn.Module:
    checkpoint = torch.load(path, weights_only=True, map_location=device)
    model = MarJazz(**checkpoint["hyperparams"])
    model.load_state_dict(checkpoint["model_state"])
    model.to(device)
    model.eval()
    print(f"Model loaded ← {path}")
    return model






Using device: cuda
Vocab size: 184


In [3]:
# ── Generate ──────────────────────────────────────────────────────────────────
def generate(
    model: nn.Module,
    seed_path: str = SEED_PATH,
    max_length: int = MAX_GEN_LEN,
    temperature: float = 0.95,
    top_k: int = 20,
    repetition_penalty: float = 1.3,
    repetition_window: int = 50
):
    model.eval()
    score = symusic.Score(seed_path)
    generated = cast(TokSequence, tokenizer.encode(score)[0]).ids[:]
 
    with torch.no_grad():
        for _ in range(max_length):
            inputs  = torch.tensor(generated).unsqueeze(0).to(device)
            outputs = model(inputs)
            logits  = outputs[0, -1]  # Raw logits for last position
 
            # 1. Repetition penalty — reduce score of recently used tokens
            recent = generated[-repetition_window:]
            for token_id in set(recent):
                if logits[token_id] > 0:
                    logits[token_id] /= repetition_penalty
                else:
                    logits[token_id] *= repetition_penalty
 
            # 2. Temperature — controls randomness (higher = more creative)
            logits = logits / temperature
 
            # 3. Top-k filtering — only keep the k most likely tokens
            if top_k > 0:
                top_k_values, _ = torch.topk(logits, top_k)
                min_top_k = top_k_values[-1]
                logits[logits < min_top_k] = float('-inf')
 
            # 4. Sample from the distribution (instead of argmax)
            probs = torch.softmax(logits, dim=0)
            next_token = int(torch.multinomial(probs, 1).item())
 
            generated.append(next_token)
 
    return generated

def save_midi(tokens, output_path: str = OUT_PATH):
    score = tokenizer.decode([TokSequence(ids=tokens)])
    score.dump_midi(output_path)
    print(f"MIDI saved → {output_path}")


In [4]:
# ── Run ───────────────────────────────────────────────────────────────────────
dataloader = load_pretokenized()
model      = MarJazz(vocab_size).to(device)
train(model, dataloader)
save_model(model)

tokens = generate(model)
save_midi(tokens)

Dataset loaded: 7809 chunks of 1024 tokens
Epoch 1/400  loss=3.2398  lr=1.00e-03
Epoch 2/400  loss=2.6983  lr=1.00e-03
Epoch 3/400  loss=2.6278  lr=1.00e-03
Epoch 4/400  loss=2.5827  lr=1.00e-03
Epoch 5/400  loss=2.5537  lr=1.00e-03
Epoch 6/400  loss=2.5289  lr=1.00e-03
Epoch 7/400  loss=2.5068  lr=1.00e-03
Epoch 8/400  loss=2.4909  lr=1.00e-03
Epoch 9/400  loss=2.4770  lr=1.00e-03
Epoch 10/400  loss=2.4665  lr=1.00e-03
Epoch 11/400  loss=2.4553  lr=1.00e-03
Epoch 12/400  loss=2.4469  lr=1.00e-03
Epoch 13/400  loss=2.4397  lr=1.00e-03
Epoch 14/400  loss=2.4335  lr=1.00e-03
Epoch 15/400  loss=2.4272  lr=1.00e-03
Epoch 16/400  loss=2.4200  lr=1.00e-03
Epoch 17/400  loss=2.4149  lr=1.00e-03
Epoch 18/400  loss=2.4118  lr=1.00e-03
Epoch 19/400  loss=2.4056  lr=1.00e-03
Epoch 20/400  loss=2.4001  lr=1.00e-03
Epoch 21/400  loss=2.3959  lr=1.00e-03
Epoch 22/400  loss=2.3907  lr=1.00e-03
Epoch 23/400  loss=2.3881  lr=1.00e-03
Epoch 24/400  loss=2.3854  lr=1.00e-03
Epoch 25/400  loss=2.3818  lr=

In [5]:
tokens = generate(model, max_length = 1000)
save_midi(tokens)

MIDI saved → /kaggle/working/generated.mid


In [6]:
tokens = generate(
    model,
    max_length = 2000,
    temperature = 0.5,
    top_k = 10, 
    repetition_penalty = 1.2,
    repetition_window = 50,
)
save_midi(tokens)

MIDI saved → /kaggle/working/generated.mid
